## Transfer Learning with ResNet-50

ResNet-50 is a deep residual network pretrained on ImageNet.
By freezing the pretrained layers and training a custom classification head,
the model leverages rich visual features while reducing training time and overfitting.


In [1]:
# Cell 1: Imports
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

In [2]:
# Cell 2: Load MNIST data
X_all = np.load(r"MNIST_Dataset/X_mnist_all.npy")
y_all = np.load(r"MNIST_Dataset/y_mnist_all.npy")

# Use a subset to prevent memory issues during development (remove or increase as needed)
X_all = X_all[:10000]
y_all = y_all[:10000]

# Add channel dimension for grayscale images: (N, 28, 28) → (N, 28, 28, 1)
X_all = np.expand_dims(X_all, axis=-1)

print("Loaded data shape:", X_all.shape, y_all.shape)

Loaded data shape: (10000, 28, 28, 1) (10000,)


In [3]:
# Cell 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (9000, 28, 28, 1) (9000,)
Test set: (1000, 28, 28, 1) (1000,)


In [4]:
# Cell 4: Preprocessing function for ResNet50
def preprocess_for_resnet(image, label):
    # Resize to 224×224 (ResNet input size)
    image = tf.image.resize(image, (224, 224))
    # Convert grayscale to RGB by repeating channels
    image = tf.image.grayscale_to_rgb(image)
    # Apply ResNet50-specific preprocessing (ImageNet mean/std subtraction)
    image = preprocess_input(image)
    return image, label

In [5]:
# Cell 5: Create tf.data pipelines
BATCH_SIZE = 16  # Small batch size to manage memory

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.map(
    preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(buffer_size=1000)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.map(
    preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE
)
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

print("Datasets prepared with batch size:", BATCH_SIZE)

Datasets prepared with batch size: 16


In [6]:
# Cell 6: One-hot encode labels
num_classes = len(np.unique(y_all))
y_train_onehot = to_categorical(y_train, num_classes)
y_test_onehot = to_categorical(y_test, num_classes)

# Note: The dataset already contains original integer labels, which are suitable for training.
# One-hot versions are kept only if needed elsewhere.

In [7]:
# Cell 7: Load ResNet50 base model (pre-trained on ImageNet)
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers for transfer learning
for layer in base_model.layers:
    layer.trainable = False

print("ResNet50 base model loaded and frozen.")

ResNet50 base model loaded and frozen.


In [8]:
# Cell 8: Build the full model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
predictions = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # Suitable for integer labels in the dataset
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
# Cell 9: Train the model using the tf.data datasets
EPOCHS = 5  # Adjust as needed

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS
)

print("Training completed.")

Epoch 1/5
204/563 ━━━━━━━━━━━━━━━━━━━━ 19:46 3s/step - accuracy: 0.7038 - loss: 0.9661

In [ ]:
# Cell 10: Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")